# Backtest Comparison Notebook
This notebook uses the provided factor data to replicate the backtest logic from the web application.
It calculates Equally-Weighted (EW) and Value-Weighted (VW) returns for Long/Short strategies and computes performance metrics (Annualized Return, Volatility, Sharpe Ratio, Max Drawdown).


In [1]:
import pandas as pd
import numpy as np
import zipfile
import io

# 1. Load Data
zip_path = 'downloadable_files.zip'
with zipfile.ZipFile(zip_path, 'r') as z:
    # Find the correct paths inside the zip
    csv_files = z.namelist()
    labels_file = next(f for f in csv_files if 'finalMonthlyLabels' in f)
    ff5_file = next(f for f in csv_files if 'ff5.csv' in f and 'dist/' not in f) # prefer the non-dist one
    
    print(f"Loading {labels_file}...")
    df = pd.read_csv(z.open(labels_file))
    
    print(f"Loading {ff5_file}...")
    ff5 = pd.read_csv(z.open(ff5_file))

# Merge risk-free rate
df = df.merge(ff5[['Month', 'Rf']], on='Month', how='left')
df['Rf'] = df['Rf'].fillna(0)
print("Data loaded successfully.")


Loading Data/Factor_Data/finalMonthlyLabels_aman.csv...
Loading Data/Factor_Data/ff5.csv...
Data loaded successfully.


In [2]:
# 2. Backtest Logic Definition

MIN_FIRMS = 5

def calc_ew_vw(group):
    # Calculate EW and VW returns for a single month's portfolio
    rets = group['Monthly_Return']
    weights = group['prev_Size'].fillna(0)
    
    ew_ret = rets.mean()
    if weights.sum() > 0:
        vw_ret = (rets * weights).sum() / weights.sum()
    else:
        vw_ret = ew_ret
        
    return pd.Series({'EW': ew_ret, 'VW': vw_ret, 'N': len(group)})

def compute_metrics(rets, rfs):
    n_months = len(rets)
    if n_months == 0:
        return {"Ann_Ret (%)": 0, "Ann_Vol (%)": 0, "Sharpe": 0, "Max_DD (%)": 0}
        
    cum_prod = (1 + rets).prod()
    years = n_months / 12
    ann_ret = (cum_prod ** (1 / years)) - 1 if years > 0 and cum_prod > 0 else 0
    
    variance = rets.var(ddof=1)
    ann_vol = np.sqrt(variance * 12) if not pd.isna(variance) else 0
    
    excess_rets = rets - rfs
    mean_excess = excess_rets.mean()
    sharpe = (mean_excess * 12) / ann_vol if ann_vol > 0 else 0
    
    cumulative = (1 + rets).cumprod()
    peak = cumulative.cummax()
    drawdown = (cumulative - peak) / peak
    max_dd = drawdown.min()
    
    return {
        "Ann_Ret (%)": round(ann_ret * 100, 2),
        "Ann_Vol (%)": round(ann_vol * 100, 2),
        "Sharpe": round(sharpe, 3),
        "Max_DD (%)": round(max_dd * 100, 2)
    }

def run_backtest(df, factor_col, long_labels, short_labels, size_col=None, size_labels=None, strategy="long_short"):
    # Filter universe if size filter is applied
    if size_col and size_labels:
        base_df = df[df[size_col].isin(size_labels)].copy()
    else:
        base_df = df.copy()
        
    # Long Leg
    long_df = base_df[base_df[factor_col].isin(long_labels)]
    long_agg = long_df.groupby('Month').apply(calc_ew_vw).reset_index()
    long_agg = long_agg[long_agg['N'] >= MIN_FIRMS] # Enforce minimum firms
    
    # Short Leg
    if strategy == "long_short":
        short_df = base_df[base_df[factor_col].isin(short_labels)]
        short_agg = short_df.groupby('Month').apply(calc_ew_vw).reset_index()
        short_agg = short_agg[short_agg['N'] >= MIN_FIRMS]
    else:
        short_agg = long_agg.copy()
        short_agg['EW'] = 0
        short_agg['VW'] = 0
        
    # Combine and align months
    months = sorted(base_df['Month'].unique())
    results = []
    
    for month in months:
        l = long_agg[long_agg['Month'] == month]
        s = short_agg[short_agg['Month'] == month]
        rf = df[df['Month'] == month]['Rf'].iloc[0] if not df[df['Month'] == month].empty else 0
        
        if not l.empty and (strategy == "long_only" or not s.empty):
            l_ew = l['EW'].values[0]
            l_vw = l['VW'].values[0]
            s_ew = s['EW'].values[0] if not s.empty else 0
            s_vw = s['VW'].values[0] if not s.empty else 0
            
            net_ew = l_ew - s_ew if strategy == "long_short" else l_ew
            net_vw = l_vw - s_vw if strategy == "long_short" else l_vw
            
            # Clamp between -200% and +200% as in backtest-core.js PORT_CAP = 2
            net_ew = max(-2, min(2, net_ew))
            net_vw = max(-2, min(2, net_vw))
            
            results.append({
                'Month': month,
                'EW_Ret': net_ew,
                'VW_Ret': net_vw,
                'Rf': rf
            })
            
    res_df = pd.DataFrame(results)
    
    if res_df.empty:
        return res_df, {}
        
    # Compute Metrics
    ew_metrics = compute_metrics(res_df['EW_Ret'], res_df['Rf'] if strategy == "long_short" else 0 * res_df['Rf'])
    vw_metrics = compute_metrics(res_df['VW_Ret'], res_df['Rf'] if strategy == "long_short" else 0 * res_df['Rf'])
    
    return res_df, {'EW': ew_metrics, 'VW': vw_metrics}


In [ ]:
# 3. Run Example: Momentum Factor (Long Winners 'W', Short Losers 'L')
# We will use Size_Label_Monthly if we want to filter by Size, or None for All.

factor = 'Momentum_Label'
long_leg = ['W']
short_leg = ['L']

res_df, metrics = run_backtest(df, factor_col=factor, long_labels=long_leg, short_labels=short_leg, strategy="long_short")

print(f"--- Momentum (W - L) ---")
print("Equally-Weighted Metrics:")
for k, v in metrics['EW'].items():
    print(f"  {k}: {v}")
    
print("\nValue-Weighted Metrics:")
for k, v in metrics['VW'].items():
    print(f"  {k}: {v}")


In [ ]:
# 4. Run Example: Value Factor (Long Value 'V', Short Growth 'G')
# Let's filter for Big caps only (Size_Label == 'B')

factor = 'BM_Label'
long_leg = ['V']
short_leg = ['G']

res_df, metrics = run_backtest(
    df, 
    factor_col=factor, 
    long_labels=long_leg, 
    short_labels=short_leg, 
    size_col='Size_Label', 
    size_labels=['B'], 
    strategy="long_short"
)

print(f"--- Value in Big Caps (V - G) ---")
print("Equally-Weighted Metrics:")
for k, v in metrics['EW'].items():
    print(f"  {k}: {v}")
    
print("\nValue-Weighted Metrics:")
for k, v in metrics['VW'].items():
    print(f"  {k}: {v}")
